# NumCompute demo — sorting, ranking and statistics (Member B)

This part of the demo shows the three modules `sort_search`, `rank` and `stats`:

1. **sort_search** — stable / multi-key sort, top-k, quickselect, binary search
2. **rank** — ranking with ties, percentiles
3. **stats** — mean / variance, axis-wise statistics, NaN handling, histogram, quantiles, streaming stats

Running example: exam scores of 5 students, then a small *students × subjects* table.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # project root, so `numcompute` can be imported

import numpy as np
from numcompute import sort_search, rank, stats

np.set_printoptions(precision=3, suppress=True)
scores = np.array([85, 92, 78, 92, 60])
scores

## 1. Sorting and searching (`sort_search`)

### 1.1 Stable sort
Equal values keep their original order — also in **descending** mode. The two 92s are at positions 1 and 3; after sorting, 1 still comes before 3.

In [ ]:
vals, idx = sort_search.stable_sort(scores, descending=True, return_indices=True)
print('sorted :', vals)
print('indices:', idx)

### 1.2 Multi-key sort
Sort students by **class (ascending)**, then by **score (descending)** inside each class.

In [ ]:
table = np.array([[2, 70],
                  [1, 88],
                  [2, 95],
                  [1, 60]])   # columns: class, score
sort_search.multikey_sort(table, keys=[0, 1], descending=[False, True])

### 1.3 Top-k
Uses `np.argpartition` (O(n)) and only sorts the k selected values: O(n + k log k) instead of O(n log n).
NaN is never preferred.

In [ ]:
top_vals, top_idx = sort_search.topk(scores, 3)
print('top-3 values :', top_vals)
print('their indices:', top_idx)
print('bottom-2     :', sort_search.topk(scores, 2, largest=False, return_indices=False))

with_nan = np.array([1.0, np.nan, 3.0, 2.0])
print('top-2 with NaN in data:', sort_search.topk(with_nan, 2, return_indices=False))

### 1.4 Quickselect
k-th smallest element (0-based) with a random pivot and a 3-way split (`<`, `==`, `>`), so duplicates and all-equal arrays are handled efficiently. Expected O(n).

In [ ]:
for k in range(len(scores)):
    print(f'k={k}:', sort_search.quickselect(scores, k, seed=0))
print('matches np.sort:', np.sort(scores))
print('all-equal array :', sort_search.quickselect(np.full(1000, 7), 500))

### 1.5 Binary search
Returns the **insertion index** and whether the value **exists**. Vectorised over many queries.

In [ ]:
sorted_scores = np.sort(scores)        # [60 78 85 92 92]
print('search 85 :', sort_search.binary_search(sorted_scores, 85))
print('search 80 :', sort_search.binary_search(sorted_scores, 80))
idx, found = sort_search.binary_search(sorted_scores, [60, 61, 92, 100])
print('many queries -> index:', idx, ' found:', found)

## 2. Ranking and percentiles (`rank`)

### 2.1 Ranking with ties
Smallest value gets rank 1. The two 92s are tied:

In [ ]:
for m in ['average', 'min', 'max', 'dense', 'ordinal']:
    print(f'{m:8s}', rank.rank(scores, method=m))

NaN gets rank NaN and does not affect the other ranks. Ranking also works along an axis.

In [ ]:
print(rank.rank([3.0, np.nan, 1.0, 3.0]))

X = np.array([[3, 1, 2],
              [1, 1, 5]])
print(rank.rank(X, method='min', axis=1))   # rank inside each row

### 2.2 Percentiles
Position `p = q/100 × (n−1)` in the sorted data; the interpolation rule decides what happens between two values. Results agree with `np.percentile`.

In [ ]:
for method in ['linear', 'lower', 'higher', 'midpoint', 'nearest']:
    ours = rank.percentile(scores, 60, interpolation=method)
    ref = np.percentile(scores, 60, method=method)
    print(f'{method:8s} ours={ours:6.2f}   numpy={ref:6.2f}')

print('quartiles:', rank.percentile(scores, [25, 50, 75]))

## 3. Descriptive statistics (`stats`)

### 3.1 Basic statistics

In [ ]:
print('mean  :', stats.mean(scores))
print('median:', stats.median(scores))
print('std   :', round(stats.std(scores), 3), '(population, ddof=0)')
print('std   :', round(stats.std(scores, ddof=1), 3), '(sample, ddof=1)')
print('min/max:', stats.minimum(scores), stats.maximum(scores))

### 3.2 Axis semantics and NaN handling
A table of 4 students × 3 subjects; one student missed an exam (NaN).

- `axis=0` → one value **per subject** (shape `(3,)`)
- `axis=1` → one value **per student** (shape `(4,)`)
- `nan_policy`: `'propagate'` (default) / `'omit'` / `'raise'`

In [ ]:
grades = np.array([[85, 70, 90],
                   [92, np.nan, 88],
                   [78, 65, 95],
                   [60, 80, 72]])
print('per subject, propagate:', stats.mean(grades, axis=0))
print('per subject, omit     :', stats.mean(grades, axis=0, nan_policy='omit'))
print('per student, omit     :', stats.mean(grades, axis=1, nan_policy='omit'))
try:
    stats.mean(grades, nan_policy='raise')
except ValueError as e:
    print('raise ->', e)

`describe` gives a full summary per column (like `pandas.DataFrame.describe`, without pandas).

In [ ]:
summary = stats.describe(grades, axis=0, nan_policy='omit')
for key, value in summary.items():
    print(f'{key:6s}', value)

### 3.3 Histogram and quantiles
Bins are `[a, b)` except the last one, which includes its right edge (same as `np.histogram`).

In [ ]:
counts, edges = stats.histogram(scores, bins=4)
print('edges :', edges)
print('counts:', counts)

rng = np.random.default_rng(0)
sample = rng.normal(loc=70, scale=10, size=1000)
counts, edges = stats.histogram(sample, bins=8)
for c, lo, hi in zip(counts, edges[:-1], edges[1:]):
    print(f'[{lo:5.1f}, {hi:5.1f})  {"#" * (c // 10)} {c}')

print('quantiles 0.1/0.5/0.9:', stats.quantile(sample, [0.1, 0.5, 0.9]))

### 3.4 Streaming statistics (Welford / Chan merge)
Data arrives in chunks (e.g. a file too big for memory). `RunningStats` keeps only count, mean and M2, and stays accurate even with a huge offset (here mean ≈ 1,000,000).

In [ ]:
big = rng.normal(loc=1e6, scale=1.0, size=100_000)
rs = stats.RunningStats(ddof=1)
for chunk in np.array_split(big, 10):
    rs.update(chunk)
print('streaming : mean=%.6f  var=%.6f  n=%d' % (rs.mean, rs.var, rs.count))
print('all at once: mean=%.6f  var=%.6f' % (big.mean(), big.var(ddof=1)))

## 4. Error messages
Invalid input gives a clear exception instead of a silent wrong answer.

In [ ]:
bad_calls = [
    lambda: sort_search.topk(scores, 10),
    lambda: sort_search.quickselect([], 0),
    lambda: rank.rank(scores, method='best'),
    lambda: rank.percentile(scores, 150),
    lambda: stats.mean(np.ones((2, 2)), axis=5),
]
for call in bad_calls:
    try:
        call()
    except Exception as e:
        print(f'{type(e).__name__}: {e}')